# 🐻 Bear Detector — Google Colab Notebook

End-to-end pipeline: **detection → tracking → segmentation → evaluation**

| Step | Description |
|------|-------------|
| 1 | Setup: clone repo, install deps, check GPU |
| 2 | Load configuration |
| 3 | Train YOLOv8 detection model |
| 4 | Detect bears in a sample image |
| 5 | Detect + track bears in a video |
| 6 | Train and run segmentation |
| 7 | Evaluate: mAP, PR curve, MOTA |
| 8 | Browse MLflow experiment logs |

> **GPU recommended.** Go to `Runtime → Change runtime type → T4 GPU` before running.


## 1 · Setup

In [ ]:
# Clone the repository
import os

REPO = 'Bear-Detector'
if not os.path.isdir(REPO):
    !git clone https://github.com/danort92/Bear-Detector.git

%cd {REPO}

In [ ]:
# Install dependencies (takes ~2 min on first run)
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from src.utils.device import get_device
from src.utils.seed import set_seed

DEVICE = get_device('auto')
set_seed(42)

print(f'🖥  Device  : {DEVICE}')
print(f'🔥 PyTorch : {torch.__version__}')
if DEVICE == 'cuda':
    print(f'🎮 GPU     : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2 · Load Configuration

In [ ]:
from src.utils.config import load_merged_config
import yaml

cfg = load_merged_config('config/default.yaml')

# ── Override anything you like here ──────────────────────────
cfg['training']['detection']['epochs']      = 20    # increase for better results
cfg['training']['detection']['batch_size']  = 16
cfg['experiment']['seed']                   = 42
cfg['mlflow']['enabled']                    = False  # set True to use MLflow UI
# ─────────────────────────────────────────────────────────────

print(yaml.dump(cfg, default_flow_style=False))

## 3 · Train YOLOv8 Detection Model

The dataset is the 1,172-image Roboflow bear-face dataset already included in the repo.
Training with `yolov8n` (nano) takes ~5–10 min on a T4 GPU for 20 epochs.

In [ ]:
from src.training.train_detection import DetectionTrainer
from src.training.experiment_tracker import ExperimentTracker
from src.utils.seed import set_seed

set_seed(cfg['experiment']['seed'])

tracker = ExperimentTracker(cfg)

with tracker.start_run() as run:
    run.log_params({
        'architecture' : cfg['model']['detection']['architecture'],
        'epochs'       : cfg['training']['detection']['epochs'],
        'lr'           : cfg['training']['detection']['learning_rate'],
        'seed'         : cfg['experiment']['seed'],
    })
    trainer = DetectionTrainer(cfg)
    output  = trainer.train()

BEST_PT = output['best_weights']
print(f'\n✅ Training done. Best weights → {BEST_PT}')

## 4 · Detect Bears in an Image

In [ ]:
# Pick a sample image from the test split
from pathlib import Path
import random

test_images = sorted(Path('Bear detection.v3i.yolov8-obb/test/images').glob('*.jpg'))
SAMPLE_IMG  = str(random.choice(test_images))
print(f'Sample image: {SAMPLE_IMG}')

In [ ]:
from src.inference.detector import BearDetector
from IPython.display import display
from PIL import Image
import cv2

detector = BearDetector(
    weights_path    = BEST_PT,
    conf_threshold  = cfg['inference']['detection_conf_threshold'],
    iou_threshold   = cfg['inference']['detection_iou_threshold'],
    device          = DEVICE,
)

result = detector.predict_image(SAMPLE_IMG, annotate=True)

print(f'Detections : {len(result["boxes"])}')
for i, (box, score, label) in enumerate(
        zip(result['boxes'], result['scores'], result['labels'])):
    print(f'  [{i}] {label}  conf={score:.3f}  box={[round(v,1) for v in box]}')

# Display annotated image
annotated_rgb = cv2.cvtColor(result['annotated_image'], cv2.COLOR_BGR2RGB)
display(Image.fromarray(annotated_rgb).resize((640, 480)))

## 5 · Detect + Track Bears in a Video

Upload your own video **or** use the synthetic test video created below.

In [ ]:
# Option A — upload your own video
from google.colab import files
import shutil

USE_UPLOAD = False  # set True to upload your own video

if USE_UPLOAD:
    uploaded = files.upload()
    INPUT_VIDEO = list(uploaded.keys())[0]
else:
    # Option B — create a tiny synthetic test video
    import numpy as np
    INPUT_VIDEO = '/tmp/test_bear_video.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(INPUT_VIDEO, fourcc, 10.0, (640, 480))
    for _ in range(30):
        frame = np.random.randint(60, 200, (480, 640, 3), dtype=np.uint8)
        writer.write(frame)
    writer.release()
    print(f'Created synthetic test video: {INPUT_VIDEO}')

print(f'Input video: {INPUT_VIDEO}')

In [ ]:
from src.tracking.sort_tracker import SORTTracker

OUTPUT_VIDEO = 'outputs/videos/bear_tracked.mp4'

tracker_sort = SORTTracker(
    max_age       = cfg['tracking']['max_age'],
    min_hits      = cfg['tracking']['min_hits'],
    iou_threshold = cfg['tracking']['iou_threshold'],
)

summary = detector.process_video(
    video_path  = INPUT_VIDEO,
    output_path = OUTPUT_VIDEO,
    tracker     = tracker_sort,
)
print(summary)

In [ ]:
# Download the annotated video
from google.colab import files
files.download(OUTPUT_VIDEO)

## 6 · Instance Segmentation

Requires a YOLO-seg format dataset. Skip training and load pre-trained COCO weights to demo the inference style.

In [ ]:
from src.segmentation.infer_segmentation import BearSegmentor

# Use COCO-pretrained yolov8n-seg to demonstrate the segmentor interface.
# For bear-specific segmentation replace 'yolov8n-seg.pt' with your trained weights.
segmentor = BearSegmentor(
    weights_path   = 'yolov8n-seg.pt',   # downloads COCO weights automatically
    conf_threshold = 0.25,
    device         = DEVICE,
)

seg_result = segmentor.predict_image(SAMPLE_IMG, annotate=True)

print(f'Instances  : {len(seg_result["boxes"])}')
print(f'Masks      : {len(seg_result["masks"])}')

seg_rgb = cv2.cvtColor(seg_result['annotated_image'], cv2.COLOR_BGR2RGB)
display(Image.fromarray(seg_rgb).resize((640, 480)))

### Train segmentation from scratch

To train with your own YOLO-seg dataset, update the config and run:

In [ ]:
# Uncomment and set your dataset path:
# cfg['data']['segmentation']['data_dir'] = 'data/processed/segmentation'
# cfg['training']['segmentation']['epochs'] = 50

# from src.segmentation.train_segmentation import SegmentationTrainer
# seg_trainer = SegmentationTrainer(cfg)
# seg_output  = seg_trainer.train()
# print(seg_output)

print('Uncomment the lines above once you have a YOLO-seg dataset ready.')

## 7 · Evaluation

In [ ]:
# Validate detection model on the test split
from src.training.train_detection import DetectionTrainer
import json

det_trainer = DetectionTrainer(cfg)
metrics = det_trainer.validate(BEST_PT)

print('\n=== Detection Metrics ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# Demo tracking metrics on synthetic data
import numpy as np
from src.tracking.metrics import compute_tracking_metrics

def _make(box_id_list):
    return [np.array([*b, tid]) for b, tid in box_id_list]

gt   = [_make([([10,10,50,50], 1)]), _make([([12,12,52,52], 1)])]
pred = [_make([([11,11,49,49], 1)]), _make([([13,13,51,51], 1)])]

trk_metrics = compute_tracking_metrics(gt, pred, iou_threshold=0.5)

print('\n=== Tracking Metrics (synthetic demo) ===')
for k, v in trk_metrics.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# View the PR curve saved during training
from pathlib import Path
from IPython.display import display
from PIL import Image
import glob

pr_files = list(Path('outputs/metrics').glob('*pr_curve*.png')) + \
           list(Path('outputs/metrics').glob('*precision_recall*.png'))

if pr_files:
    display(Image.open(pr_files[0]))
else:
    print('No PR curve found yet — run evaluate_detection() first.')

## 8 · MLflow Experiment Tracking

MLflow is disabled by default in Colab (set `cfg['mlflow']['enabled'] = True` above to enable it).
When enabled, you can browse runs with:

In [ ]:
# Start the MLflow tracking server in the background
# Requires cfg['mlflow']['enabled'] = True during training.

# !mlflow ui --backend-store-uri outputs/experiments/mlruns --port 5000 &

# Then use ngrok or Colab port forwarding to access the UI:
# from google.colab.output import eval_js
# print(eval_js("google.colab.kernel.proxyPort(5000)"))

# Or simply print experiment JSON logs:
import json
from pathlib import Path

log_files = list(Path('outputs/experiments').glob('*.json'))
if log_files:
    with open(log_files[0]) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print('No experiment logs yet. Set mlflow.enabled=False to use JSON logging.')

## Tips for Colab

| Tip | How |
|-----|-----|
| Save your trained weights | `from google.colab import files; files.download(BEST_PT)` |
| Use Google Drive | `from google.colab import drive; drive.mount('/content/drive')` |
| Monitor GPU usage | `!nvidia-smi` |
| Check YOLO training live | Open `outputs/models/detection/*/results.csv` |
| Resume training | Set `pretrained_weights` in config to a previous `.pt` file |

---

**GitHub:** https://github.com/danort92/Bear-Detector  
**Author:** Danilo Ortelli
